<a href="https://colab.research.google.com/github/JuanMauwu/udea-ai4eng-20252-saberpro/blob/main/05%20-%20modelo%20xgboost%20classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROYECTO KAGGLE: Modelo adicional - xgboost classifier**
---

## **Descarga y preparación del conjunto de datos**

In [1]:
import os

os.environ['KAGGLE_CONFIG_DIR'] = '.'
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia
!unzip udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip

  0% 0.00/29.9M [00:00<?, ?B/s]
100% 29.9M/29.9M [00:00<00:00, 1.01GB/s]
Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  inflating: submission_example.csv  
  inflating: test.csv                
  inflating: train.csv               


## **Modelo completo**

In [2]:
#importaciones

import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib

#Carga de datos

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")
submission_example = pd.read_csv("submission_example.csv")

#Guardamos los IDs del test para el archivo final
test_ids = df_test['ID']

#Preprocesamiento unificado en unica funcipon

#Definimos una función para aplicar la misma limpieza a Train y Test
def preprocesar_data(df, es_train=True):
    df = df.copy()

    #Normalizar columnas
    df.columns = df.columns.str.strip().str.lower()

    #Eliminar columnas irrelevantes
    drop_cols = ['periodo_academico', 'id']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    #Limpieza de textos
    df = df.map(lambda x: str(x).strip().lower() if isinstance(x, str) else x)
    df = df.replace({'sí': 'si', 'n': 'no'})

    #Mapeos Ordinales (Tu lógica del notebook 02)
    map_matricula = {
        'menos de 2.5 millones': 1,
        'entre 2.5 millones y menos de 4 millones': 2,
        'entre 4 millones y menos de 5.5 millones': 3,
        'entre 5.5 millones y menos de 7 millones': 4,
        'más de 7 millones': 5
    }
    if 'e_valormatriculauniversidad' in df.columns:
        df['e_valormatriculauniversidad'] = df['e_valormatriculauniversidad'].map(map_matricula)

    map_horas = {
        '0': 0, 'menos de 10 horas': 1, 'entre 11 y 20 horas': 2,
        'entre 21 y 30 horas': 3, 'más de 30 horas': 4
    }
    if 'e_horassemanatrabaja' in df.columns:
        df['e_horassemanatrabaja'] = df['e_horassemanatrabaja'].map(map_horas)

    map_estrato = {
        'estrato 1': 1, 'estrato 2': 2, 'estrato 3': 3,
        'estrato 4': 4, 'estrato 5': 5, 'estrato 6': 6
    }
    if 'f_estratovivienda' in df.columns:
        df['f_estratovivienda'] = df['f_estratovivienda'].map(map_estrato)

    #Imputación de Nulos
    #Numéricas
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for col in num_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    #Categóricas
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if col != 'rendimiento_global': # No tocar el target si existe
            if df[col].isna().any():
                if df[col].dropna().isin(['si', 'no']).any():
                    df[col] = df[col].fillna('no')
                else:
                    df[col] = df[col].fillna('desconocido')

    #Codificación Binaria
    binary_cols = [c for c in df.columns if df[c].dropna().isin(['si', 'no']).all()]
    for c in binary_cols:
        df[c] = df[c].map({'si': 1, 'no': 0})

    #Educacion Padres
    educ_map = {
        'ninguno': 0, 'primaria completa': 1, 'secundaria (bachillerato) completa': 2,
        'técnica o tecnológica incompleta': 3, 'técnica o tecnológica completa': 4,
        'universitario': 5, 'postgrado': 6, 'no sabe': 0, 'desconocido': 0
    }
    for col in ['f_educacionpadre', 'f_educacionmadre']:
        if col in df.columns:
            df[col] = df[col].map(educ_map).fillna(0) # Llenar nans remanentes con 0

    return df

#Procesamos ambos
print("Preprocesando Train y Test")
X_train_clean = preprocesar_data(df_train, es_train=True)
X_test_clean = preprocesar_data(df_test, es_train=False)

#Separar Target
target_col = 'rendimiento_global'
y = X_train_clean[target_col]
X = X_train_clean.drop(columns=[target_col])

#Ajuste
#XGBoost expects numerical labels for multi-class classification
le = LabelEncoder()
y_encoded = le.fit_transform(y)


#One-Hot Encoding
#Importante: Usamos pd.get_dummies y luego alineamos las columnas
print("Aplicando One-Hot Encoding")
#Aplique get_dummies primero a columnas de programa específicas
X = pd.get_dummies(X, columns=['e_prgm_academico', 'e_prgm_departamento'], drop_first=True)
X_test_final = pd.get_dummies(X_test_clean, columns=['e_prgm_academico', 'e_prgm_departamento'], drop_first=True)

#Identifique las columnas de objetos restantes después del OHE específico inicial
#paso crucial para capturar cualquier columna categórica que se haya omitido.
remaining_object_cols_X = X.select_dtypes(include=['object']).columns
remaining_object_cols_X_test = X_test_final.select_dtypes(include=['object']).columns

#Combinarlos para garantizar la coherencia y aplique get_dummies a todos.
#El uso de 'set' garantiza nombres de columnas únicos y 'list' convierte nuevamente a una lista para get_dummies.
all_remaining_object_cols = list(set(remaining_object_cols_X) | set(remaining_object_cols_X_test))

if all_remaining_object_cols: # Only apply if there are actual columns to process
    print(f"Aplicando One-Hot Encoding a columnas categóricas restantes: {all_remaining_object_cols}")
    X = pd.get_dummies(X, columns=all_remaining_object_cols, drop_first=True)
    X_test_final = pd.get_dummies(X_test_final, columns=all_remaining_object_cols, drop_first=True)

#Alineación de columnas (Para que Test tenga las mismas columnas que Train)
#Rellena con 0 las columnas que falten en Test y elimina las que sobren
X, X_test_final = X.align(X_test_final, join='left', axis=1, fill_value=0)

#Escalado
print("Escalando variables numéricas")
schaler = StandardScaler()
cols_to_scale = ['indicador_1', 'indicador_2', 'indicador_3', 'indicador_4']
#Asegurarse que existen
cols_to_scale = [c for c in cols_to_scale if c in X.columns]

X[cols_to_scale] = schaler.fit_transform(X[cols_to_scale])
X_test_final[cols_to_scale] = schaler.transform(X_test_final[cols_to_scale])


#MODELO


print("Entrenando Modelo")

#Configuramos el model XGBoost Classifier

clf = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_), #Specify number of classes
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    eval_metric='merror',
    random_state=42,
    n_jobs=-1
)

cv_results = cross_validate(clf, X, y_encoded, cv=2, scoring='accuracy', return_train_score=True)

print(f"Resultados CV - Train Score: {np.mean(cv_results['train_score']):.4f}")
print(f"Resultados CV - Test Score (Validación): {np.mean(cv_results['test_score']):.4f}")

#Entrenamiento Final con todos los datos
clf.fit(X, y_encoded) # Use y_encoded
print("Modelo entrenado exitosamente.")

#Generación de submission

print("Generando predicciones para Kaggle")
predictions_encoded = clf.predict(X_test_final)
predictions = le.inverse_transform(predictions_encoded)

#Crear DataFrame de Submission
submission = pd.DataFrame({
    'ID': test_ids,
    'RENDIMIENTO_GLOBAL': predictions
})

#Verificar formato
print(submission.head())
print(f"Dimensiones: {submission.shape}")

#Guardar
submission.to_csv("xgbBoost.csv", index=False)
print("Archivo guardado. ¡Listo para subir a Kaggle!")

Preprocesando Train y Test
Aplicando One-Hot Encoding
Aplicando One-Hot Encoding a columnas categóricas restantes: ['e_privado_libertad']
Escalando variables numéricas
Entrenando Modelo
Resultados CV - Train Score: 0.4343
Resultados CV - Test Score (Validación): 0.4120
Modelo entrenado exitosamente.
Generando predicciones para Kaggle
       ID RENDIMIENTO_GLOBAL
0  550236               bajo
1   98545         medio-alto
2  499179               alto
3  782980               bajo
4  785185               bajo
Dimensiones: (296786, 2)
Archivo guardado. ¡Listo para subir a Kaggle!
